In [1]:
!pip install -q "crisperwhisper[transformers]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.1/107.1 kB 1.9 MB/s eta 0:00:00a 0:00:01


In [2]:
from crisperwhisper import CrisperWhisperModel

model = CrisperWhisperModel("small", backend="transformers")

added_tokens.json:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/356 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  484MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] WhisperForConditionalGeneration LOAD REPORT from: nyralabs/CrisperWhisper2.0_small
Key                       | Status     |  | 
--------------------------+------------+--+-
encoder_blank_head.bias   | UNEXPECTED |  | 
encoder_blank_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

In [3]:
import os, glob

folder = "/kaggle/input/datasets/pavithra1mj/sample-dataset"
exts = (".mp3", ".wav", ".m4a", ".mp4", ".flac", ".ogg", ".aac")
files = [f for f in glob.glob(folder + "/**/*", recursive=True) if f.lower().endswith(exts)]
print(files)

AUDIO = files[0]

!ffmpeg -y -i "$AUDIO" -t 60 -vn -ac 1 -ar 16000 -c:a pcm_s16le /kaggle/working/test.wav

result = model.transcribe("/kaggle/working/test.wav", language="en", word_timestamps=True)
print(result.text)

for w in result.words[:30]:
    print(f"{w.start:6.2f}-{w.end:6.2f}  {w.word}")

['/kaggle/input/datasets/pavithra1mj/sample-dataset/audio.mp3.mp3']
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --ena

[transformers] The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens', 'output_attentions'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization

When you get stuck and don't remember the right word, instead of just staying silent, you can actually use these three philophresses. Number one, what's that thing called? Ria, can you pass me that [UM] what's that thing called, a big spoon with holes? Oh, you mean a sieve. Sure, I'll get it. [breath] Number two, it's on the tip of my tongue. [yawn] What's the word? It's on the tip of my tongue. When the team gets together to throw around ideas. You mean brainstorming? Number three. You know what I mean? The boss is like [UM] you know you know what I mean. [UM] yes, he's like. Tough-minded. Yeah, I get it. Try using these phrases instead of staying silent.
  0.34-  0.46  When
  0.46-  0.58  you
  0.58-  0.72  get
  1.04-  1.20  stuck
  1.52-  1.69  and
  1.69-  1.94  don't
  2.22-  2.52  remember
  2.74-  2.84  the
  2.84-  3.06  right
  3.26-  3.52  word,
  4.20-  4.30  instead
  4.30-  4.42  of
  4.64-  4.94  just
  5.24-  5.34  staying
  5.72-  6.08  silent,
  6.62-  6.71  you
  6.7

In [4]:
import re

words = [(w.word.strip(), w.start, w.end) for w in result.words]
tag = lambda t: t.startswith("[") and t.endswith("]")
clean = lambda t: re.sub(r"[^a-z0-9']", "", t.lower())

FILLER_TAGS = {"um", "uh", "er", "erm", "hmm", "mm", "ah"}
PHRASES = {("you", "know"), ("i", "mean"), ("kind", "of"), ("sort", "of")}

fillers = [(t, round(s, 2)) for t, s, e in words if tag(t) and clean(t) in FILLER_TAGS]
events = [(t, round(s, 2)) for t, s, e in words if tag(t) and clean(t) not in FILLER_TAGS]

tw = [(clean(t), s, e) for t, s, e in words if not tag(t) and clean(t)]

phrase_fillers = [(" ".join((tw[i][0], tw[i + 1][0])), round(tw[i][1], 2)) for i in range(len(tw) - 1) if (tw[i][0], tw[i + 1][0]) in PHRASES]
word_reps = [(tw[i][0], round(tw[i][1], 2)) for i in range(len(tw) - 1) if tw[i][0] == tw[i + 1][0]]
phrase_reps = [(" ".join((tw[i][0], tw[i + 1][0])), round(tw[i][1], 2)) for i in range(len(tw) - 3) if (tw[i][0], tw[i + 1][0]) == (tw[i + 2][0], tw[i + 3][0])]

pauses = [(round(words[i][2], 2), round(words[i + 1][1] - words[i][2], 2)) for i in range(len(words) - 1) if words[i + 1][1] - words[i][2] >= 0.5]

minutes = (tw[-1][2] - tw[0][1]) / 60

print("WPM:", round(len(tw) / minutes))
print("Filler sounds:", len(fillers), fillers)
print("Filler phrases:", len(phrase_fillers), phrase_fillers)
print("Word repetitions:", len(word_reps), word_reps)
print("Phrase repetitions:", len(phrase_reps), phrase_reps)
print("Vocal events:", events)
print("Pauses >= 0.5s:", len(pauses), pauses)

WPM: 159
Filler sounds: 3 [('[UM]', 14.18), ('[UM]', 34.98), ('[UM]', 37.82)]
Filler phrases: 5 [('you know', 32.6), ('i mean', 33.3), ('you know', 35.72), ('you know', 36.96), ('i mean', 37.46)]
Word repetitions: 0 []
Phrase repetitions: 1 [('you know', 35.72)]
Vocal events: [('[breath]', 20.16), ('[yawn]', 23.48)]
Pauses >= 0.5s: 13 [(3.52, 0.68), (6.08, 0.54), (9.12, 1.12), (10.74, 0.6), (14.5, 0.98), (19.42, 0.74), (20.24, 0.72), (21.36, 0.6), (28.86, 0.54), (31.84, 0.76), (34.98, 0.74), (35.96, 0.88), (43.64, 0.52)]
